# Ackland–Jones Structure Classification

The Ackland–Jones method (Ackland & Jones, 2006) classifies each atom as **FCC**, **BCC**, **HCP**, **ICO** (icosahedral), or **other/unknown** using a decision tree on the *chi-parameter* histogram — a histogram of cosine angles between all pairs of neighbors.

In [ ]:
import pyscal3
from pyscal3.structures import make_crystal
import numpy as np
from collections import Counter

## Perfect Crystals

For ideal lattices every atom is correctly identified.

In [ ]:
for name in ["fcc", "bcc", "hcp"]:
    lc = {"fcc": 4.05, "bcc": 2.87, "hcp": 3.21}[name]
    atoms = make_crystal(name, lattice_constant=lc, repetitions=(4, 4, 4))
    pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
    labels, names = pyscal3.identify_ackland_jones(atoms)
    counts = Counter(names)
    print(f"{name.upper():>4} → {dict(counts)}")

## Effect of Thermal Noise

Adding Gaussian noise simulates thermal vibrations.  FCC remains well-classified at moderate noise levels.

In [ ]:
print("{:<8} {:>6} {:>6} {:>6} {:>6}".format("Noise", "FCC", "HCP", "BCC", "other"))
print("-" * 34)

for noise in [0.0, 0.02, 0.05, 0.1]:
    atoms = make_crystal("fcc", lattice_constant=4.05,
                         repetitions=(4, 4, 4), noise=noise)
    pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
    labels, _ = pyscal3.identify_ackland_jones(atoms)
    n = len(atoms)
    c = Counter(labels.tolist())
    print(f"{noise:<8.2f} {c.get(1,0):>6} {c.get(2,0):>6} "
          f"{c.get(3,0):>6} {c.get(0,0):>6}")

## Accessing Results

Results are stored as per-atom labels in `atoms.arrays["pyscal_ackland_label"]` and string names in `atoms.arrays["pyscal_structure"]`.

In [ ]:
atoms = make_crystal("hcp", lattice_constant=3.21, repetitions=(4, 4, 4))
pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
labels, names = pyscal3.identify_ackland_jones(atoms)

print("Labels array shape :", labels.shape)
print("Unique labels      :", np.unique(labels))
print("Structure names    :", list(set(names)))
print("Stored array keys  :", [k for k in atoms.arrays if "ackland" in k or "structure" in k])

## Underlying Chi Parameters

The classifier uses the 9-bin chi-parameter histogram.
You can inspect the raw chi vectors:

In [ ]:
for name in ["fcc", "bcc", "hcp"]:
    lc = {"fcc": 4.05, "bcc": 2.87, "hcp": 3.21}[name]
    atoms = make_crystal(name, lattice_constant=lc, repetitions=(3, 3, 3))
    pyscal3.find_neighbors(atoms, method="cutoff", cutoff=0)
    chi = pyscal3.chi_params(atoms)
    print(f"{name.upper()} chi: {chi[0].tolist()}")

## References

1. G. J. Ackland and A. P. Jones, "Applications of local crystal structure measures in experiment and simulation", *Phys. Rev. B* **73**, 054104 (2006). [doi:10.1103/PhysRevB.73.054104](https://doi.org/10.1103/PhysRevB.73.054104)